In [4]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import make_scorer, mean_squared_error

warnings.filterwarnings("ignore")

DATA_DIR = Path("data")

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test_x.csv")

TARGET = "bilissel_performans_skoru"

X = train.drop(columns=["id", TARGET])
y = train[TARGET]

test_ids = test["id"]
X_test = test.drop(columns=["id"])

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include="object").columns.tolist()

print("Train:", X.shape)
print("Test:", X_test.shape)
print("Numeric columns:", len(num_cols))
print("Categorical columns:", len(cat_cols))

Train: (56000, 22)
Test: (24000, 22)
Numeric columns: 15
Categorical columns: 7


In [5]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Bilinmiyor")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    max_depth=None
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

In [6]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

rmse_scorer = make_scorer(rmse, greater_is_better=False)

cv = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    pipeline,
    X,
    y,
    cv=cv,
    scoring=rmse_scorer,
    n_jobs=-1
)

rmse_scores = -scores

print("Fold RMSE:", rmse_scores)
print("Mean RMSE:", rmse_scores.mean())
print("Std RMSE:", rmse_scores.std())

Fold RMSE: [1.2816447  1.28348368 1.27256982 1.29351856 1.30528642]
Mean RMSE: 1.287300635393191
Std RMSE: 0.011186657457849148


## Deney 2 - Median Imputation + Missing Indicator

In [7]:
numeric_transformer_exp2 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True))
])

categorical_transformer_exp2 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Bilinmiyor")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_exp2 = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_exp2, num_cols),
        ("cat", categorical_transformer_exp2, cat_cols)
    ]
)

model_exp2 = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    max_depth=None
)

pipeline_exp2 = Pipeline(steps=[
    ("preprocessor", preprocessor_exp2),
    ("model", model_exp2)
])

scores_exp2 = cross_val_score(
    pipeline_exp2,
    X,
    y,
    cv=cv,
    scoring=rmse_scorer,
    n_jobs=-1
)

rmse_scores_exp2 = -scores_exp2

print("Fold RMSE:", rmse_scores_exp2)
print("Mean RMSE:", rmse_scores_exp2.mean())
print("Std RMSE:", rmse_scores_exp2.std())

Fold RMSE: [1.28208303 1.28363382 1.27253663 1.29350328 1.30437916]
Mean RMSE: 1.2872271839573095
Std RMSE: 0.010851417954733506


## Deney 3 - Log Transform + Missing Indicator

In [8]:
X_exp3 = X.copy()
X_test_exp3 = X_test.copy()

log_cols = [
    "uyku_oncesi_kafein_mg",
    "uyku_oncesi_ekran_suresi_dk"
]

for col in log_cols:
    if col in X_exp3.columns:
        X_exp3[col] = np.log1p(X_exp3[col])
        X_test_exp3[col] = np.log1p(X_test_exp3[col])

numeric_transformer_exp3 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True))
])

categorical_transformer_exp3 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Bilinmiyor")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_exp3 = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_exp3, num_cols),
        ("cat", categorical_transformer_exp3, cat_cols)
    ]
)

model_exp3 = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    max_depth=None
)

pipeline_exp3 = Pipeline(steps=[
    ("preprocessor", preprocessor_exp3),
    ("model", model_exp3)
])

scores_exp3 = cross_val_score(
    pipeline_exp3,
    X_exp3,
    y,
    cv=cv,
    scoring=rmse_scorer,
    n_jobs=-1
)

rmse_scores_exp3 = -scores_exp3

print("Fold RMSE:", rmse_scores_exp3)
print("Mean RMSE:", rmse_scores_exp3.mean())
print("Std RMSE:", rmse_scores_exp3.std())

Fold RMSE: [1.28203055 1.28351529 1.27253212 1.29357373 1.30427411]
Mean RMSE: 1.287185160843363
Std RMSE: 0.010840623409594984


## Deney 4 - Feature Engineering + Log Transform + Missing Indicator

In [9]:
X_exp4 = X.copy()
X_test_exp4 = X_test.copy()

# Log dönüşümü
log_cols = [
    "uyku_oncesi_kafein_mg",
    "uyku_oncesi_ekran_suresi_dk"
]

for col in log_cols:
    if col in X_exp4.columns:
        X_exp4[col] = np.log1p(X_exp4[col])
        X_test_exp4[col] = np.log1p(X_test_exp4[col])


def add_features(df):
    df = df.copy()

    # Stres ve çalışma yükü
    if "stres_skoru" in df.columns and "gunluk_calisma_saati" in df.columns:
        df["stres_calisma_yuku"] = df["stres_skoru"] * df["gunluk_calisma_saati"]

    # Uyku kalitesi göstergesi
    if "rem_yuzdesi" in df.columns and "derin_uyku_yuzdesi" in df.columns:
        df["uyku_kalitesi_orani"] = df["rem_yuzdesi"] + df["derin_uyku_yuzdesi"]

    # Uyku bozulma skoru
    if "gecelik_uyanma_sayisi" in df.columns and "uykuya_dalma_suresi_dk" in df.columns:
        df["uyku_bozulma_skoru"] = df["gecelik_uyanma_sayisi"] + df["uykuya_dalma_suresi_dk"]

    # Dijital yük
    if "uyku_oncesi_ekran_suresi_dk" in df.columns and "uyku_oncesi_kafein_mg" in df.columns:
        df["dijital_kafein_yuku"] = df["uyku_oncesi_ekran_suresi_dk"] * df["uyku_oncesi_kafein_mg"]

    # Aktivite / stres dengesi
    if "gunluk_adim_sayisi" in df.columns and "stres_skoru" in df.columns:
        df["aktivite_stres_orani"] = df["gunluk_adim_sayisi"] / (df["stres_skoru"] + 1)

    return df


X_exp4 = add_features(X_exp4)
X_test_exp4 = add_features(X_test_exp4)

num_cols_exp4 = X_exp4.select_dtypes(include=np.number).columns.tolist()
cat_cols_exp4 = X_exp4.select_dtypes(include="object").columns.tolist()

numeric_transformer_exp4 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True))
])

categorical_transformer_exp4 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Bilinmiyor")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_exp4 = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_exp4, num_cols_exp4),
        ("cat", categorical_transformer_exp4, cat_cols_exp4)
    ]
)

model_exp4 = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    max_depth=None
)

pipeline_exp4 = Pipeline(steps=[
    ("preprocessor", preprocessor_exp4),
    ("model", model_exp4)
])

scores_exp4 = cross_val_score(
    pipeline_exp4,
    X_exp4,
    y,
    cv=cv,
    scoring=rmse_scorer,
    n_jobs=-1
)

rmse_scores_exp4 = -scores_exp4

print("Fold RMSE:", rmse_scores_exp4)
print("Mean RMSE:", rmse_scores_exp4.mean())
print("Std RMSE:", rmse_scores_exp4.std())

Fold RMSE: [1.27703181 1.27639545 1.26436099 1.28405915 1.29810131]
Mean RMSE: 1.2799897415709276
Std RMSE: 0.011050026622092169


## Deney 5 - Feature Engineering + HistGradientBoostingRegressor

In [10]:
from sklearn.ensemble import HistGradientBoostingRegressor

model_exp5 = HistGradientBoostingRegressor(
    max_iter=500,
    learning_rate=0.05,
    max_leaf_nodes=31,
    l2_regularization=0.1,
    random_state=42
)

pipeline_exp5 = Pipeline(steps=[
    ("preprocessor", preprocessor_exp4),
    ("model", model_exp5)
])

scores_exp5 = cross_val_score(
    pipeline_exp5,
    X_exp4,
    y,
    cv=cv,
    scoring=rmse_scorer,
    n_jobs=-1
)

rmse_scores_exp5 = -scores_exp5

print("Fold RMSE:", rmse_scores_exp5)
print("Mean RMSE:", rmse_scores_exp5.mean())
print("Std RMSE:", rmse_scores_exp5.std())

Fold RMSE: [1.22842344 1.22863866 1.21706492 1.22660698 1.24400267]
Mean RMSE: 1.2289473334773597
Std RMSE: 0.008645253496770222


## Deney 6 - HistGradientBoosting Hyperparameter Denemesi

In [11]:
model_exp6 = HistGradientBoostingRegressor(
    max_iter=800,
    learning_rate=0.03,
    max_leaf_nodes=45,
    min_samples_leaf=20,
    l2_regularization=0.05,
    random_state=42
)

pipeline_exp6 = Pipeline(steps=[
    ("preprocessor", preprocessor_exp4),
    ("model", model_exp6)
])

scores_exp6 = cross_val_score(
    pipeline_exp6,
    X_exp4,
    y,
    cv=cv,
    scoring=rmse_scorer,
    n_jobs=-1
)

rmse_scores_exp6 = -scores_exp6

print("Fold RMSE:", rmse_scores_exp6)
print("Mean RMSE:", rmse_scores_exp6.mean())
print("Std RMSE:", rmse_scores_exp6.std())

Fold RMSE: [1.22964944 1.22862504 1.21702951 1.22701673 1.24524317]
Mean RMSE: 1.2295127784213464
Std RMSE: 0.009058856132983253


## Deney 7 - HistGradientBoosting daha sade tuning

In [12]:
model_exp7 = HistGradientBoostingRegressor(
    max_iter=700,
    learning_rate=0.04,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    l2_regularization=0.1,
    random_state=42
)

pipeline_exp7 = Pipeline(steps=[
    ("preprocessor", preprocessor_exp4),
    ("model", model_exp7)
])

scores_exp7 = cross_val_score(
    pipeline_exp7,
    X_exp4,
    y,
    cv=cv,
    scoring=rmse_scorer,
    n_jobs=-1
)

rmse_scores_exp7 = -scores_exp7

print("Fold RMSE:", rmse_scores_exp7)
print("Mean RMSE:", rmse_scores_exp7.mean())
print("Std RMSE:", rmse_scores_exp7.std())

Fold RMSE: [1.22849956 1.2290441  1.21597071 1.22432274 1.24502272]
Mean RMSE: 1.228571966718488
Std RMSE: 0.00946258977464743


## Deney 7 ile Submission Üretimi

In [14]:
best_pipeline = pipeline_exp7

best_pipeline.fit(X_exp4, y)

test_preds = best_pipeline.predict(X_test_exp4)

submission = pd.DataFrame({
    "id": test_ids,
    TARGET: test_preds
})

submission.head()

submission_clipped = submission.copy()

submission_clipped[TARGET] = submission_clipped[TARGET].clip(0, 10)

print(submission_clipped.shape)
print(submission_clipped.head())
print(submission_clipped[TARGET].describe())

submission.to_csv("submission_exp7_hgb.csv", index=False)

(24000, 2)
   id  bilissel_performans_skoru
0   1                   5.837882
1   2                   6.839400
2   3                   3.182555
3   4                   7.189681
4   5                   3.609035
count    24000.000000
mean         5.939861
std          1.850152
min          0.132316
25%          4.635788
50%          6.031417
75%          7.339575
max         10.000000
Name: bilissel_performans_skoru, dtype: float64
